# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第7章 産出実験 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。

**生成される図・CSVの保存場所について。** コード中に出てくる
`data/processed/`や`assets/figures/`は、あなたのGoogle Driveの中の
フォルダではない。Colabの仮想マシン上に一時的に作られるだけの作業用
フォルダであり、**セッションが切断される(一定時間の無操作、ランタイムの
再起動など)と中身ごと消える。** 保存したい図やCSVがあれば、Colab画面
左側のファイル一覧からその都度手元のパソコンにダウンロードするか、
コードを`/content/drive/...`のようなGoogle Drive上のパスに書き出すよう
自分で書き換える必要がある。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch07_recording_setup.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質管理」節の模式図。防音環境における話者・
ヘッドセットマイク・録音機器の配置を、上から見た模式図として示す。
概念図であり、実際の部屋の実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

elements <- tibble(
  x = c(5, 5, 8.5),
  y = c(3, 3.35, 3),
  label = c("話者", "ヘッドセットマイク\n(口から一定距離)", "録音機器\n(PC/レコーダー)"),
  type = c("speaker", "mic", "recorder")
)

room <- tibble(xmin = 0, xmax = 10, ymin = 0, ymax = 6)

p <- ggplot() +
  geom_rect(data = room, aes(xmin = xmin, xmax = xmax, ymin = ymin, ymax = ymax),
            fill = "#F4F1EA", colour = "#8C8C8C", linewidth = 1) +
  annotate("text", x = 0.3, y = 5.7, label = "防音・準防音環境", hjust = 0, size = 3.2, colour = "grey30", fontface = "italic") +
  geom_point(aes(x = 5, y = 3), size = 6, colour = "#34518A") +
  geom_point(aes(x = 5, y = 3.35), shape = 17, size = 4, colour = "#A8681F") +
  geom_segment(aes(x = 5, xend = 5, y = 3.05, yend = 3.3), linetype = 2, colour = "grey40", linewidth = 0.4) +
  annotate("text", x = 5.15, y = 3.17, label = "一定距離", size = 2.6, colour = "grey40", hjust = 0) +
  geom_point(aes(x = 8.5, y = 3), shape = 15, size = 5, colour = "#5C7A3D") +
  geom_segment(aes(x = 5.1, xend = 8.3, y = 3.35, yend = 3.1),
               arrow = arrow(length = unit(0.15, "cm"), type = "closed"), colour = "grey50", linewidth = 0.4) +
  annotate("text", x = 5, y = 2.6, label = "話者", size = 3, colour = "#1B2333") +
  annotate("text", x = 5, y = 3.7, label = "ヘッドセットマイク", size = 2.8, colour = "#1B2333") +
  annotate("text", x = 8.5, y = 2.6, label = "録音機器", size = 3, colour = "#1B2333") +
  coord_fixed(xlim = c(-0.3, 10.3), ylim = c(-0.3, 6.3)) +
  theme_void() +
  labs(caption = "概念図(上から見た模式図)。scripts/R/ch07_recording_setup.R より生成。") +
  theme(plot.caption = element_text(size = rel(0.8), colour = "grey35"))

save_fig(p, "fig07-02-recording-setup", chapter = "ch07", width = 6.5, height = 4)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_mic_distance_snr.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質管理」節の作例。マイクロフォンと話者の口との
距離が変動すると、音圧レベル(逆二乗則に基づく音圧の距離減衰)が
変化し、信号対雑音比(SNR)がどれだけ変わるかを、実際に計算して
確認する。理論的な計算例であり、実測データではない。
音圧は音源からの距離にほぼ反比例する(自由音場を仮定した理論値)

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

ref_distance <- 0.05     # 基準距離(ヘッドセットマイクの典型的な距離、5cm)
ref_spl_db   <- 70       # 基準距離での音圧レベル(dB SPL、仮定値)
noise_floor_db <- 35     # 録音環境の暗騒音レベル(dB SPL、仮定値)

distances <- c(0.05, 0.10, 0.30, 0.50)   # メートル: ヘッドセット, 近接, デスクマイク, 離れた設置

spl_at_distance <- ref_spl_db - 20 * log10(distances / ref_distance)
snr_at_distance <- spl_at_distance - noise_floor_db

d <- data.frame(distance_m = distances,
                spl_db = round(spl_at_distance, 1),
                snr_db = round(snr_at_distance, 1))
print(d)


## `ch07_clipping_detection.R`

対象章: 第7章「産出実験」

「録音環境・機材・音質管理」節の作例。クリッピング(振幅が記録可能な
上限を超えて飽和する現象)を、波形のサンプル値から自動的に検出する
簡便な方法を、架空の波形データで確認する。実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

fs <- 16000
dur <- 1
t <- seq(0, dur, length.out = fs * dur)

make_waveform <- function(gain) {
  raw <- gain * (sin(2 * pi * 150 * t) + 0.3 * sin(2 * pi * 450 * t) + rnorm(length(t), 0, 0.05))
  pmin(pmax(raw, -1), 1)   # -1から1の範囲で飽和(クリッピング)させる、16bit相当の正規化波形
}

detect_clipping <- function(wave, threshold = 0.999, min_run = 3) {
  is_ceiling <- abs(wave) >= threshold
  # 連続run長を計算し、min_run以上連続して天井に張り付いている箇所を検出する
  runs <- rle(is_ceiling)
  clipped_samples <- sum(runs$lengths[runs$values & runs$lengths >= min_run])
  clipped_samples / length(wave)
}

gains <- c(0.5, 0.9, 1.3, 2.0)
results <- map_dfr(gains, function(g) {
  wave <- make_waveform(g)
  tibble(gain = g, pct_clipped = detect_clipping(wave) * 100)
})

print(results)

d_plot <- map_dfr(gains, function(g) tibble(gain = g, t = t[1:800], amp = make_waveform(g)[1:800]))
p <- ggplot(d_plot, aes(t, amp)) +
  geom_line(colour = "#34518A", linewidth = 0.3) +
  geom_hline(yintercept = c(-1, 1), linetype = 2, colour = "#A8681F") +
  facet_wrap(~gain, labeller = label_both, nrow = 1) +
  labs(x = "時間 (s)", y = "振幅(正規化)",
       caption = "シミュレーション波形。破線は記録可能な振幅の上限・下限。scripts/R/ch07_clipping_detection.R より生成。")
save_fig(p, "fig07-04-clipping-detection", chapter = "ch07", width = 8, height = 2.6)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_lombard_diagnosis.R`

対象章: 第7章「産出実験」

「産出実験に固有のバイアス」節の作例。実験の丁寧発話(Lombard効果に
類する順応)が、試行順序に伴ってどう減衰していくかを、母音空間の
広さ(F1・F2から計算した凸包面積)を指標に、架空のデータで確認する。
実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_trials  <- 80
# 丁寧発話の度合いは、実験開始直後にもっとも強く、指数関数的に減衰して
# 話者本来の(くだけた)発話パターンに近づいていくと想定する。
politeness_decay <- 25 * exp(-(1:n_trials) / 20)
vowel_space_area <- 180 + politeness_decay + rnorm(n_trials, 0, 8)

d <- tibble(trial = 1:n_trials, area = vowel_space_area)

m <- lm(area ~ trial, data = d)
slope <- coef(m)["trial"]
p_val <- summary(m)$coefficients["trial", "Pr(>|t|)"]

early_mean <- mean(d$area[d$trial <= 10])
late_mean  <- mean(d$area[d$trial > (n_trials - 10)])

cat("試行順の係数(1試行あたりの母音空間面積の変化):", round(slope, 3), "\n")
cat("p値:", format.pval(p_val, digits = 3), "\n")
cat("最初の10試行の平均面積:", round(early_mean, 1), "\n")
cat("最後の10試行の平均面積:", round(late_mean, 1), "\n")
cat("減少率:", round((early_mean - late_mean) / early_mean * 100, 1), "%\n")

p <- ggplot(d, aes(trial, area)) +
  geom_point(alpha = 0.5, size = 1.5, colour = "#34518A") +
  geom_smooth(method = "lm", se = TRUE, colour = "#A8681F", linewidth = 0.8, formula = y ~ x) +
  labs(x = "試行番号", y = "母音空間の面積(模式値)",
       caption = "シミュレーションデータ。scripts/R/ch07_lombard_diagnosis.R より生成。")
save_fig(p, "fig07-03-lombard-decay", chapter = "ch07", width = 6, height = 3.8)
cat("完了。図はassets/figures/ch07/に保存した。\n")


## `ch07_practice_trial_exclusion.R`

対象章: 第7章「産出実験」

「産出実験に固有のバイアス」節の作例。丁寧発話の減衰(ch07_lombard_
diagnosis.Rと同じ生成モデル)が残っている区間を含めたまま平均を
取ると、外れ値を除いた後も系統的なバイアスが残ることを、最初の
練習試行を除外した場合と除外しない場合とで比較して確認する。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

set.seed(20260906)

n_trials <- 80
politeness_decay <- 25 * exp(-(1:n_trials) / 20)
vowel_space_area <- 180 + politeness_decay + rnorm(n_trials, 0, 8)
d <- tibble(trial = 1:n_trials, area = vowel_space_area)

true_baseline <- 180   # 丁寧発話の影響がなくなった後の真の値

n_practice <- 15
mean_all       <- mean(d$area)
mean_excluded  <- mean(d$area[d$trial > n_practice])

cat("真のベースライン(丁寧発話の影響なし):", true_baseline, "\n")
cat("全80試行の平均(練習試行を除外しない場合):", round(mean_all, 1),
    "  真値からの乖離:", round(mean_all - true_baseline, 1), "\n")
cat("最初の", n_practice, "試行を練習試行として除外した場合の平均:",
    round(mean_excluded, 1), "  真値からの乖離:",
    round(mean_excluded - true_baseline, 1), "\n")
